<a href="https://colab.research.google.com/github/ProfAI/machine-learning-fondamenti/blob/main/Progetto%20Finale%20-%20Cross%20Selling%20di%20Polizze/health_insurance_cross_sell_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Previsione di opportunità di Cross Sell di assicurazioni

Il cliente è una compagnia di assicurazioni che ha fornito un'assicurazione sanitaria ai suoi clienti, adesso hanno bisogno del tuo aiuto per costruire un modello predittivo in grado di prevedere se gli assicurati dell'anno passato potrebbero essere interessati ad acquistare anche un'assicurazione per il proprio veicolo.

Il dataset è composto dalle seguenti proprietà:
- **id**: id univoco dell'acquirente.
- **Gender**: sesso dell'acquirente.
- **Age**: età dell'acquirente.
- **Driving_License**: 1 se l'utente ha la patente di guida, 0 altrimenti.
- **Region_Code**: codice univoco della regione dell'acquirente.
- **Previously_Insured**: 1 se l'utente ha già un veicolo assicurato, 0 altrimenti.
- **Vehicle_Age**: età del veicolo
- **Vehicle_Damage**: 1 se l'utente ha danneggiato il veicolo in passato, 0 altrimenti.
- **Annual_Premium**: la cifra che l'utente deve pagare come premio durante l'anno.
- **Policy_Sales_Channel**: codice anonimizzato del canale utilizzato per la proposta (es. per email, per telefono, di persona, ecc...)
- **Vintage**: numero di giorni dalla quale l'utente è cliente dell'azienda.
- **Response**: 1 se l'acquirente ha risposto positivametne alla proposta di vendità, 0 altrimenti.

L'obiettivo del modello è prevedere il valore di **Response**.

**Tip**
Fai attenzione alla distribuzione delle classi, dai uno sguardo a [questo approfondimento](https://machinelearningmastery.com/tactics-to-combat-imbalanced-classes-in-your-machine-learning-dataset/). In caso di classi sbilanciate puoi provare a:

- Penalizzare la classe più frequente (ricorda l'argomento class_weight)
- Utilizzare [l'oversampling o l'undersampling](https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/).


[LINK AL DATASET (Richiede un'account su Kaggle)](https://www.kaggle.com/anmolkumar/health-insurance-cross-sell-prediction)

Il **Progetto Finale** rappresenta un caso d'uso end-to-end industriale: stimare la propensione dei clienti che possiedono già una polizza sanitaria ad acquistare un'assicurazione per il proprio veicolo (cross-selling).

Questo progetto mette a frutto tutte le fasi tipiche di una pipeline reale di Data Science:
1. **Exploratory Data Analysis (EDA)**: Comprendere la distribuzione del target, le correlazioni e identificare potenziali problemi (es. forte sbilanciamento delle classi).
2. **Data Preprocessing**: Trattamento dei dati mancanti, encoding di variabili categoriche nominali/ordinali e scaling delle feature numeriche.
3. **Model Selection & Validation**: Addestramento e ottimizzazione di diversi modelli di classificazione (es. regressione logistica, alberi decisionali, ensemble) tramite tecniche robuste come la Cross-Validation.
4. **Evaluation**: Scelta delle metriche più opportune (es. ROC AUC, F1-Score) per gestire lo sbilanciamento delle classi e guidare le decisioni commerciali.


In [ ]:
import os
import kagglehub

# Configura il percorso locale del dataset già scaricato per il fallback
cached_path = "/home/rares/.cache/kagglehub/datasets/anmolkumar/health-insurance-cross-sell-prediction/versions/1"

try:
    # Tenta il download con kagglehub
    path = kagglehub.dataset_download("anmolkumar/health-insurance-cross-sell-prediction")
    print("Dataset scaricato tramite kagglehub in:", path)
except Exception as e:
    print(f"Impossibile scaricare tramite kagglehub ({e}). Uso del percorso locale di fallback...")
    if os.path.exists(cached_path):
        path = cached_path
        print("Fallback sul percorso locale del dataset:", path)
    else:
        raise FileNotFoundError("Dataset non trovato localmente. Assicurati che sia presente in ~/.cache/kagglehub.")


## 1. Analisi Esplorativa dei Dati (EDA)

Iniziamo importando le librerie fondamentali (`pandas`, `numpy`, `matplotlib`, `seaborn`) e caricando il dataset di training per comprenderne la struttura, le tipologie dei dati e verificare l'eventuale presenza di valori nulli o outlier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Impostiamo il seed per la riproducibilità didattica
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configurazione dello stile dei grafici
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Caricamento dei dataset
train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

print(f"Dimensioni del dataset di Train: {train_df.shape}")
print(f"Dimensioni del dataset di Test: {test_df.shape}")
train_df.head()


### Controllo dei Valori Mancanti e Tipi di Dati

Prima di procedere ad analisi statistiche più approfondite, è fondamentale verificare se vi sono valori mancanti (NaN) che richiederebbero tecniche di imputazione, e controllare i tipi di dati associati a ciascuna colonna.

In [ ]:
print("Informazioni generali sul dataset di Train:")
print(train_df.info())

print("\nValori mancanti per ciascuna colonna del Train:")
print(train_df.isnull().sum())


### Analisi del Sbilanciamento del Target (`Response`)

Il target del nostro modello è la colonna **Response** (1 = interessato ad acquistare l'assicurazione veicolo, 0 = non interessato).

Nei problemi reali di cross-selling o rilevamento frodi, le classi sono tipicamente **fortemente sbilanciate** (imbalanced). È essenziale misurare questa proporzione: se addestrassimo un classificatore su dati sbilanciati senza opportuni accorgimenti, il modello potrebbe massimizzare l'accuratezza banale predicendo sempre la classe maggioritaria (in questo caso `0`), risultando però del tutto inutile per le decisioni aziendali. Visualizziamo la distribuzione delle classi.

In [ ]:
response_counts = train_df['Response'].value_counts()
response_pct = train_df['Response'].value_counts(normalize=True) * 100

print("Frequenza assoluta delle classi:")
print(response_counts)
print("\nPercentuale relativa delle classi:")
print(response_pct)

plt.figure(figsize=(6, 5))
sns.countplot(x='Response', data=train_df, hue='Response', palette='viridis', legend=False)
plt.title("Distribuzione della Risposta (Target)")
plt.xlabel("Risposta (0 = Non Interessato, 1 = Interessato)")
plt.ylabel("Numero di Clienti")
plt.show()


### Relazione tra Feature Categoriche e Target

Analizziamo le principali feature categoriche per verificare l'esistenza di pattern evidenti:
- `Gender` (Sesso dell'acquirente)
- `Driving_License` (Possesso della patente)
- `Previously_Insured` (Se possiede già un veicolo assicurato)
- `Vehicle_Age` (Età del veicolo)
- `Vehicle_Damage` (Danni passati al veicolo)

In [ ]:
categorical_cols = ['Gender', 'Driving_License', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage']

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(ax=axes[i], x=col, hue='Response', data=train_df, palette='muted')
    axes[i].set_title(f"Distribuzione di {col} rispetto al Target")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequenza")

# Rimuoviamo l'ultimo subplot vuoto per motivi estetici
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


### Analisi delle Feature Numeriche Continue

Ora esaminiamo le variabili continue:
- `Age`: Età dell'assicurato.
- `Annual_Premium`: Premio annuale da pagare.
- `Vintage`: Giorni trascorsi dall'inizio del rapporto di clientela.

Visualizziamo prima le statistiche descrittive generali e poi le densità di distribuzione (KDE) condizionate alla classe target.

In [ ]:
numerical_cols = ['Age', 'Annual_Premium', 'Vintage']
train_df[numerical_cols].describe()


Le densità condizionate ci aiutano a verificare se ci sono differenze significative nella distribuzione delle feature continue tra chi risponde positivamente (`Response=1`) e chi negativamente (`Response=0`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    if col == 'Annual_Premium':
        # La feature Annual_Premium ha outlier estremi a destra. Per visualizzare
        # la densità principale limitiamo l'asse x a 100.000
        sns.kdeplot(ax=axes[i], x=col, hue='Response', data=train_df, fill=True, common_norm=False, palette='crest')
        axes[i].set_xlim(0, 100000)
    else:
        sns.kdeplot(ax=axes[i], x=col, hue='Response', data=train_df, fill=True, common_norm=False, palette='crest')
    axes[i].set_title(f"Densità di {col} per classe Target")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Densità")

plt.tight_layout()
plt.show()


## 2. Preprocessing dei Dati e Prevenzione del Data Leakage

### ⚠️ Che cos'è il Data Leakage?
Il **Data Leakage** si verifica quando le informazioni provenienti dal validation o dal test set "fuggono" nel training set durante le fasi di preprocessing. Ad esempio, se calcoliamo la media e la deviazione standard per standardizzare (`StandardScaler`) l'intero dataset *prima* di suddividerlo, la media del training conterrà informazioni del test. Questo falsa la stima dell'errore di generalizzazione del modello durante la validazione.

### Best Practices
1. **Dividere il dataset subito**: Eseguiamo il `train_test_split` prima di qualsiasi manipolazione.
2. **Fittare i transformer solo sul training**: Calcoliamo i parametri di scaling (`fit_transform`) esclusivamente su `X_train` e usiamoli per trasformare (`transform`) `X_val` e `X_test`.
3. **Encoding delle variabili categoriche**:
   - `Gender`: Mappatura binaria (`Male` -> 1, `Female` -> 0).
   - `Vehicle_Damage`: Mappatura binaria (`Yes` -> 1, `No` -> 0).
   - `Vehicle_Age`: Mappatura ordinale poiché c'è una chiara gerarchia (`< 1 Year` -> 0, `1-2 Year` -> 1, `> 2 Years` -> 2).

In [ ]:
from sklearn.model_selection import train_test_split

# Rimuoviamo la feature non informativa 'id' e separiamo il target
X = train_df.drop(columns=['id', 'Response'])
y = train_df['Response']

# Suddividiamo in Train e Validation Set (80/20) con stratificazione per preservare la proporzione delle classi
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Dimensioni di X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Dimensioni di X_val: {X_val.shape}, y_val: {y_val.shape}")


Scriviamo ora una funzione di preprocessing riutilizzabile che automatizza la codifica e lo scaling delle feature continue, assicurando di non incappare nel data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

def preprocess_data(X_data, scaler=None, is_train=True):
    X_processed = X_data.copy()
    
    # 1. Encoding Binario per Gender e Vehicle_Damage
    X_processed['Gender'] = X_processed['Gender'].map({'Male': 1, 'Female': 0})
    X_processed['Vehicle_Damage'] = X_processed['Vehicle_Damage'].map({'Yes': 1, 'No': 0})
    
    # 2. Ordinal Encoding per Vehicle_Age
    vehicle_age_map = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
    X_processed['Vehicle_Age'] = X_processed['Vehicle_Age'].map(vehicle_age_map)
    
    # 3. Standardizzazione dei valori numerici continui
    cols_to_scale = ['Age', 'Annual_Premium', 'Vintage']
    
    if is_train:
        scaler = StandardScaler()
        X_processed[cols_to_scale] = scaler.fit_transform(X_processed[cols_to_scale])
        return X_processed, scaler
    else:
        if scaler is None:
            raise ValueError("Uno scaler fittato deve essere fornito durante la fase di validazione/test.")
        X_processed[cols_to_scale] = scaler.transform(X_processed[cols_to_scale])
        return X_processed

# Applichiamo il preprocessing a Train e Validation set
X_train_preprocessed, scaler = preprocess_data(X_train, is_train=True)
X_val_preprocessed = preprocess_data(X_val, scaler=scaler, is_train=False)

print("Esempio di dati preprocessati:")
X_train_preprocessed.head()


## 3. Selezione del Modello e Valutazione delle Performance

### Come gestire lo sbilanciamento delle classi?
Per contrastare lo sbilanciamento delle classi, utilizzeremo due approcci:
1. **Pesi delle Classi Bilanciati**: Gli algoritmi scikit-learn supportano l'argomento `class_weight='balanced'`, che assegna penalità più elevate agli errori sulla classe minoritaria. La formula per il peso della classe $j$ è:
   $$w_j = \frac{N}{\text{n\_classes} \times N_j}$$
   dove $N$ è il numero totale di campioni e $N_j$ è il numero di campioni della classe $j$.
2. **Valutazione con metriche robuste**: L'accuratezza è fuorviante. Utilizzeremo:
   - **ROC AUC**: Misura la capacità globale di discriminazione a qualsiasi soglia di classificazione.
   - **F1-Score**: La media armonica di Precision e Recall.
     $$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Confronteremo tre modelli in Cross-Validation:
- **Logistic Regression** (Baseline lineare)
- **HistGradientBoostingClassifier** (Algoritmo ensemble ad alte performance, ideale per dataset massivi)
- **Random Forest Classifier** (Robusto modello ensemble basato su bagging)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# Definiamo i modelli da confrontare, testando sia la versione standard che quella bilanciata
models = {
    "Logistic Regression (Sbilanciata)": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "Logistic Regression (Bilanciata)": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
    "HistGradientBoosting (Sbilanciata)": HistGradientBoostingClassifier(random_state=RANDOM_SEED),
    "HistGradientBoosting (Bilanciata)": HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_SEED)
}

# Eseguiamo la cross-validation con 3 folds usando la metrica ROC AUC
for name, model in models.items():
    print(f"Valutazione in Cross-Validation per {name}...")
    scores = cross_val_score(model, X_train_preprocessed, y_train, cv=3, scoring='roc_auc', n_jobs=-1)
    print(f"  ROC AUC medio: {scores.mean():.4f} (+/- {scores.std():.4f})\n")


Ora addestriamo ciascun modello sull'intero `X_train_preprocessed` e ne verifichiamo la performance sul validation set per confrontare i report di classificazione e le metriche di precisione e recall dettagliate.

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train_preprocessed, y_train)
    y_pred = model.predict(X_val_preprocessed)
    y_pred_proba = model.predict_proba(X_val_preprocessed)[:, 1]
    
    auc = roc_auc_score(y_val, y_pred_proba)
    f1 = f1_score(y_val, y_pred)
    
    results[name] = {
        "y_pred": y_pred,
        "y_pred_proba": y_pred_proba,
        "ROC AUC": auc,
        "F1 Score": f1
    }
    
    print(f"=== REPORT DI CLASSIFICAZIONE PER: {name} ===")
    print(f"ROC AUC: {auc:.4f} | F1-Score: {f1:.4f}")
    print(classification_report(y_val, y_pred))
    print("-" * 60)


### Confronto con Random Forest

Aggiungiamo un modello basato su foresta casuale (`RandomForestClassifier`) bilanciata. Per contenere l'uso di risorse computazionali e velocizzare l'esecuzione, impostiamo `max_depth=12` e `n_estimators=100`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Addestramento di Random Forest (Bilanciato) in corso...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_model.fit(X_train_preprocessed, y_train)

y_pred_rf = rf_model.predict(X_val_preprocessed)
y_pred_proba_rf = rf_model.predict_proba(X_val_preprocessed)[:, 1]

auc_rf = roc_auc_score(y_val, y_pred_proba_rf)
f1_rf = f1_score(y_val, y_pred_rf)

results["Random Forest (Bilanciato)"] = {
    "y_pred": y_pred_rf,
    "y_pred_proba": y_pred_proba_rf,
    "ROC AUC": auc_rf,
    "F1 Score": f1_rf
}

print("=== REPORT DI CLASSIFICAZIONE PER: Random Forest (Bilanciato) ===")
print(f"ROC AUC: {auc_rf:.4f} | F1-Score: {f1_rf:.4f}")
print(classification_report(y_val, y_pred_rf))


## 4. Visualizzazione Grafica delle Curve di Performance

Confrontiamo graficamente le curve **ROC** e **Precision-Recall** per i tre modelli principali bilanciati. Questo ci permetterà di confrontare il comportamento dei modelli al variare delle soglie di decisione commerciale.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

plt.figure(figsize=(14, 6))

# 1. ROC Curve
plt.subplot(1, 2, 1)
for name in ["Logistic Regression (Bilanciata)", "HistGradientBoosting (Bilanciata)", "Random Forest (Bilanciato)"]:
    fpr, tpr, _ = roc_curve(y_val, results[name]["y_pred_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['ROC AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label="Classificatore Casuale (AUC = 0.500)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR / Recall)")
plt.title("Curve ROC a confronto")
plt.legend(loc="lower right")

# 2. Precision-Recall Curve
plt.subplot(1, 2, 2)
for name in ["Logistic Regression (Bilanciata)", "HistGradientBoosting (Bilanciata)", "Random Forest (Bilanciato)"]:
    precision, recall, _ = precision_recall_curve(y_val, results[name]["y_pred_proba"])
    plt.plot(recall, precision, label=f"{name} (F1 = {results[name]['F1 Score']:.3f})")
plt.xlabel("Recall (Sensibilità)")
plt.ylabel("Precision (Precisione)")
plt.title("Curve Precision-Recall a confronto")
plt.legend(loc="lower left")

plt.tight_layout()
plt.show()


### Analisi delle Matrici di Confusione

Confrontiamo la Matrice di Confusione tra la versione **Sbilanciata** (standard) e quella **Bilanciata** (pesata) di `HistGradientBoosting`. Questo contrasto evidenzierà graficamente come varia la capacità del modello di intercettare i clienti interessati.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Modello Sbilanciato
cm_unbal = confusion_matrix(y_val, results["HistGradientBoosting (Sbilanciata)"]["y_pred"])
disp_unbal = ConfusionMatrixDisplay(confusion_matrix=cm_unbal, display_labels=["Non Inter.", "Interessato"])
disp_unbal.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title("HistGradientBoosting (Sbilanciata)\nSoglia di decisione standard = 0.5")

# Modello Bilanciato
cm_bal = confusion_matrix(y_val, results["HistGradientBoosting (Bilanciata)"]["y_pred"])
disp_bal = ConfusionMatrixDisplay(confusion_matrix=cm_bal, display_labels=["Non Inter.", "Interessato"])
disp_bal.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title("HistGradientBoosting (Bilanciata)\nClass Weight bilanciati")

plt.tight_layout()
plt.show()


### 💡 Interpretazione Aziendale delle Metriche

- **Modello Sbilanciato (Soglia standard 0.5)**: Prevede quasi sempre `0` a causa del forte sbilanciamento. Sebbene l'accuratezza totale sia elevata (~88%), commette un numero enorme di falsi negativi (manca quasi tutti i clienti interessati). Dal punto di vista del business, questa soluzione non genera valore perché non individua le opportunità di cross-selling.
- **Modello Bilanciato**: Penalizzando gli errori sulla classe minoritaria, il classificatore abbassa la soglia per etichettare un cliente come `1`. Questo aumenta in modo sensibile il **Recall** (identificando la stragrande maggioranza dei potenziali clienti interessati), pur accettando una precisione più bassa (più falsi positivi, ovvero contatti a vuoto).

### Come sfruttare le probabilità predette? (Leads Prioritization)
Nello scenario industriale reale, il team di marketing non contatterà i clienti usando una decisione rigida (0/1). Invece:
1. Estrarrà la probabilità di acquisto per ogni cliente tramite `predict_proba`.
2. Ordinerà i contatti dal più propense al meno propense (ranking di decrescente probabilità).
3. Deciderà di contattare ad esempio solo il primo 20% dei clienti. Poiché il modello `HistGradientBoosting` ha un ROC AUC di ~0.85, questo top 20% conterrà la stragrande maggioranza di tutti i potenziali acquirenti, permettendo di risparmiare l'80% del budget di marketing a fronte di una perdita minima di opportunità di vendita.

In [ ]:
# Applichiamo lo stesso preprocessing del training set al test set ufficiale
# Nota: Passiamo scaler=scaler e is_train=False per prevenire data leakage!
X_test_preprocessed = preprocess_data(test_df.drop(columns=['id']), scaler=scaler, is_train=False)

# Selezioniamo il miglior modello addestrato (HistGradientBoosting Bilanciato)
best_model = models["HistGradientBoosting (Bilanciata)"]

# Generiamo probabilità e classi predette
test_predictions_proba = best_model.predict_proba(X_test_preprocessed)[:, 1]
test_predictions_binary = best_model.predict(X_test_preprocessed)

# Costruiamo il DataFrame di submission/predizioni finali
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Response_Probability': test_predictions_proba,
    'Response_Class': test_predictions_binary
})

# Salviamo le predizioni in formato CSV
output_csv_path = os.path.join(path, "test_predictions.csv")
submission_df.to_csv(output_csv_path, index=False)

print(f"Predizioni esportate con successo in: {output_csv_path}")
print(f"Dimensioni file esportato: {submission_df.shape}")
submission_df.head()
